# GPU Wavelets (JPEG2000-style MRA) — Colab driver

Runtime → Change runtime type → **GPU** (T4 is fine). Then run cells top to bottom.

This builds the project with CMake and runs the deterministic 1D/2D/MRA tests.

In [ ]:
!nvidia-smi
!nvcc --version | tail -1
!cmake --version | head -1

## Get the code onto Colab
Run the next cell and, when prompted, upload **`wavelets_project.zip`** (in your repo folder). It extracts to `/content/wavelets` and `cd`s in. If your repo is on GitLab/GitHub instead, uncomment the `git clone` line and set the URL.

In [ ]:
# Put the code on the Colab machine. If it's not already there, upload
# wavelets_project.zip (in your repo folder) when the picker appears.
# Alternative: uncomment the git clone and set your repo URL.
import os
TARGET = '/content/wavelets'
# !git clone <YOUR_REPO_URL> {TARGET}
if not os.path.exists(f'{TARGET}/CMakeLists.txt'):
    from google.colab import files
    import zipfile
    up = files.upload()                    # choose wavelets_project.zip
    os.makedirs(TARGET, exist_ok=True)
    with zipfile.ZipFile(next(iter(up))) as z:
        z.extractall(TARGET)
%cd /content/wavelets
!ls

In [ ]:
# Detect the GPU's compute capability so the build targets the right arch.
import subprocess
cc = subprocess.check_output(
    ['nvidia-smi', '--query-gpu=compute_cap', '--format=csv,noheader']
).decode().strip().split('\n')[0].replace('.', '')
print('compute capability =', cc)
!cmake -B build -DCMAKE_CUDA_ARCHITECTURES={cc} -DCMAKE_BUILD_TYPE=Release
!cmake --build build -j

## Run tests (items 1, 2, 3, 4, 8, 9, 10, 14)
Deterministic binaries — `test_1d/2d/access/3d/fp16/io`. Each returns nonzero on failure; `ctest` aggregates.

In [ ]:
!cd build && ctest --output-on-failure

## CLI demo — image I/O + round-trip (items 4, 13)
Reconstructs the castle image (round-trip) and writes the MRA representation.

In [ ]:
!cd build && ./wavelet --input ../assets/Castle_Lichtenstein.jpg --output recon.png --levels 3
!cd build && ./wavelet --input ../assets/Castle_Lichtenstein.jpg --output mra.png --levels 3 --forward
from IPython.display import Image, display
display(Image('build/recon.png'), Image('build/mra.png'))

## Benchmarks — items 5, 6, 7, 10, 11
Saves markdown tables to `bench_results.md`; paste them into `report.md`'s `FILL` slots.

In [ ]:
!cd build && ./bench | tee ../bench_results.md

## Register / shared-memory usage (item 11)
`--ptxas-options=-v` is on, so per-kernel register/smem usage prints at compile time. Force a recompile of `bench` to surface it:

In [ ]:
!cd build && touch ../bench/bench.cu && cmake --build . --target bench 2>&1 | grep -iE "ptxas info|registers|bytes smem|bytes stack"